# TD4 — Feature engineering EEG pour l'estimation de la charge cognitive

## Contexte

Ce TD s'inscrit dans le projet fil rouge basé sur le papier :

**Multimodal Brain-Computer Interface for In-Vehicle Driver Cognitive Load Measurement: Dataset and Baselines**.

Le papier introduit le dataset **CL-Drive**, dans lequel des signaux EEG, ECG, EDA et Gaze sont enregistrés pendant une tâche de conduite simulée. Les scores de charge cognitive sont collectés toutes les **10 secondes**. Après prétraitement, les signaux sont segmentés en fenêtres de 10 s, puis transformés en caractéristiques numériques pour entraîner des modèles de classification.

Dans ce TD, on se concentre uniquement sur les **features EEG**.

---

## Objectifs pédagogiques

À la fin du TD, vous devez être capables de :

1. expliquer pourquoi on transforme un signal EEG en vecteur de caractéristiques ;
2. distinguer les features temporelles, fréquentielles et non linéaires ;
3. expliquer le principe de la densité spectrale de puissance, ou PSD ;
4. calculer des puissances par bande EEG ;
5. interpréter l'entropie spectrale ;
6. expliquer les paramètres de Hjorth ;
7. comprendre le principe de la complexité de Lempel-Ziv ;
8. comprendre l'idée de la dimension fractale de Higuchi ;
9. structurer une fonction complète d'extraction de features EEG ;
10. préparer une matrice de features pour la classification.

---

## Features EEG ciblées

Le papier regroupe les features EEG suivantes :

| Famille | Features |
|---|---|
| PSD | puissance absolue, moyenne, maximale, minimale et médiane |
| Entropie spectrale | entropie calculée à partir de la PSD normalisée |
| Hjorth | mobility et complexity |
| Lempel-Ziv | complexité d'une séquence binarisée |
| Higuchi | dimension fractale |
| Statistiques temporelles | moyenne, minimum, maximum, médiane, variance, écart-type |

Dans ce TD, on adopte une version complète :

- PSD calculée dans les cinq bandes EEG : delta, theta, alpha, beta, gamma ;
- entropie spectrale calculée dans les cinq bandes ;
- features non linéaires calculées sur le segment temporel ;
- statistiques temporelles calculées sur le segment.

On obtient donc :

$$
5 \text{ bandes} \times 5 \text{ descripteurs PSD} = 25
$$

$$
5 \text{ entropies spectrales} = 5
$$

$$
2 \text{ Hjorth} + 1 \text{ Lempel-Ziv} + 1 \text{ Higuchi} + 6 \text{ statistiques} = 10
$$

Soit au total :

$$
25 + 5 + 10 = 40 \text{ features par canal EEG}
$$

## Questions de compréhension

### Question 1

Pourquoi ne donne-t-on pas directement le signal EEG brut à un classifieur classique comme LDA, SVM ou Random Forest ?

### Réponse 

Le signal EEG brut est trop dense, il est a 256Hz donc 2560 valeurs sur 10 secondes, en plus les modèles ne peuvent pas utiliser la temporalité du signal (c'est un seul vecteur pas une suite de vecteurs).
Il y a aussi trop de variations entre les sujets sans traitement des données. Il faut se servir de la baseline de chacun pour normaliser les données.

### Question 2

Pourquoi les features fréquentielles sont-elles particulièrement importantes en EEG ?

### Réponse 

Parce que chaque bande est associée à un état cognitif. Le tableau suivant résume les états cognitifs par bande:

| Bande | Fréquence | Évolution avec la charge cognitive |
|---|---|---|
| Delta | 0.5–4 Hz | Pas de lien direct avec la charge cognitive |
| Theta | 4–8 Hz | Augmente avec la sollicitation de la mémoire de travail |
| Alpha | 8–12 Hz | Diminue avec la sortie de l'état de repos cognitif |
| Beta | 12–30 Hz | Augmente avec l'effort mental actif |
| Gamma | 30–75 Hz | Augmente avec l'effort musculaire, moins utile car très bruité |

### Question 3

Pourquoi faut-il calculer les features séparément sur chaque canal EEG ?

### Réponse 

Parce que chaque canal est lié à l'activité d'une zone différente du cerveau.  
AF7, AF8 -> frontal -> mémoire de travail  
TP9, TP10 -> temporal -> auditif, mémoire  
Si on mélange tout, on perd l'information d'où se passe l'activité.

## 1. Bandes fréquentielles EEG

Les signaux EEG sont souvent analysés par bandes de fréquence.

| Bande | Intervalle utilisé dans ce TD | Interprétation générale |
|---|---:|---|
| Delta | 0.5–4 Hz | activité lente |
| Theta | 4–8 Hz | attention, mémoire de travail, somnolence selon contexte |
| Alpha | 8–12 Hz | relaxation, inhibition, yeux fermés |
| Beta | 12–30 Hz | activité mentale, attention, activité motrice |
| Gamma | 30–75 Hz | activité rapide, intégration, mais sensible aux artefacts musculaires |

Dans le papier, la bande gamma va jusqu'à 75 Hz. 

### Question 4

Pourquoi peut-on limiter la bande gamma à 45 Hz dans certaines implémentations ?

### Réponse 

Au dessus de 45Hz il y a beaucoup d'artefacts musculaires (contractions des muscles du visage, de la mâchoire ou du cou). Ils génèrent des signaux électriques dans les mêmes fréquences que le gamma haut.  
On peut couper à 45 Hz pour ne pas confondre du bruit musculaire avec de l'activité cérébrale.

## 2. Méthodologie d'implémentation 

Dans ce TD, l'objectif est de comprendre puis implémenter les fonctions essentielles.

Pour chaque fonction, vous aurez :

- une explication théorique ;
- l'algorithme ;
- les fonctions Python recommandées ;
- les paramètres importants ;
- des questions de vérification.

Les bibliothèques utiles sont :

| Objectif | Bibliothèque | Fonctions utiles |
|---|---|---|
| Tableaux numériques | NumPy | `np.array`, `np.mean`, `np.var`, `np.diff`, `np.median` |
| Données tabulaires | pandas | `pd.DataFrame`, `pd.read_csv`, `to_csv` |
| PSD | scipy.signal | `welch` |
| Entropie | scipy.stats | `entropy` |
| Visualisation | matplotlib | `plt.plot`, `plt.semilogy`, `plt.bar` |

### Travail à réaliser

Vous devez construire progressivement les fonctions suivantes :

1. `compute_psd_band_features(signal, fs)` ;
2. `compute_spectral_entropy_bands(signal, fs)` ;
3. `compute_hjorth(signal)` ;
4. `lempel_ziv_complexity(signal)` ;
5. `higuchi_fd(signal, kmax=10)` ;
6. `compute_raw_features(signal)` ;
7. `extract_eeg_features(signal, fs)` ;
8. une boucle permettant d'extraire les features sur des fenêtres de 10 secondes.

In [3]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import entropy

FS = 256
WINDOW_SEC = 10
WINDOW_SAMPLES = FS * WINDOW_SEC

EEG_BANDS = {
    "delta": (0.5, 4),
    "theta": (4, 8),
    "alpha": (8, 12),
    "beta": (12, 31),
    "gamma": (31, 75),
}

print("Nombre d'échantillons par fenêtre :", WINDOW_SAMPLES)
print("Bandes EEG :", EEG_BANDS)

Nombre d'échantillons par fenêtre : 2560
Bandes EEG : {'delta': (0.5, 4), 'theta': (4, 8), 'alpha': (8, 12), 'beta': (12, 31), 'gamma': (31, 75)}


## 3. Exemples pédagogiques sur signaux connus

Avant d'appliquer les features à l'EEG réel, on commence par des signaux connus.

On va comparer :

1. une sinusoïde à 2 Hz, principalement dans la bande delta ;
2. une sinusoïde à 10 Hz, principalement dans la bande alpha ;
3. une sinusoïde à 20 Hz, principalement dans la bande beta ;
4. un bruit blanc, dont l'énergie est plus répartie ;
5. un mélange de plusieurs composantes.

### Objectif pédagogique

Vérifier que les features donnent des résultats cohérents :

- une sinusoïde pure doit avoir une PSD concentrée ;
- le bruit doit avoir une entropie spectrale plus élevée ;
- une sinusoïde alpha doit avoir une puissance alpha dominante.

In [4]:
# Cellule d'aide : génération de signaux synthétiques pour les tests pédagogiques.
# Vous pouvez l'utiliser pour tester vos futures fonctions de features.

def generate_synthetic_signals(fs=FS, duration=10, random_state=0):
    rng = np.random.default_rng(random_state)
    t = np.arange(0, duration, 1/fs)
    signals = {
        "delta_2Hz": np.sin(2*np.pi*2*t),
        "alpha_10Hz": np.sin(2*np.pi*10*t),
        "beta_20Hz": np.sin(2*np.pi*20*t),
        "white_noise": rng.normal(0, 1, size=len(t)),
        "mixed": 0.8*np.sin(2*np.pi*6*t) + 0.5*np.sin(2*np.pi*10*t) + 0.2*rng.normal(0, 1, size=len(t)),
    }
    return t, signals

t, synthetic_signals = generate_synthetic_signals()

## 4. Feature PSD : densité spectrale de puissance

### Principe théorique

La **PSD**, ou densité spectrale de puissance, indique comment la puissance du signal est répartie selon la fréquence.

Pour un segment EEG $x[n]$, on estime la PSD avec la méthode de Welch. L'idée est de :

1. découper le signal en sous-fenêtres ;
2. calculer un spectre sur chaque sous-fenêtre ;
3. moyenner les spectres pour obtenir une estimation plus stable.

En Python, on utilise généralement :

```python
freqs, psd = scipy.signal.welch(signal, fs=fs, nperseg=...)
```

### Paramètres recommandés

- `fs=256` pour CL-Drive ;
- `nperseg=fs*2` pour une fenêtre Welch de 2 secondes ;
- si le segment est court, prendre `nperseg=min(len(signal), fs*2)` ;
- sélectionner ensuite les fréquences appartenant à une bande donnée à l’aide d’un masque booléen (utiliser le vecteur `freqs` retourné `scipy.signal.welch`).

### Features PSD pour chaque bande

Pour chaque bande EEG, calculer :

1. puissance absolue : somme de la PSD dans la bande ;
2. puissance moyenne ;
3. puissance maximale ;
4. puissance minimale ;
5. puissance médiane.

### Algorithme

1. calculer `freqs, psd` avec Welch ;

Pour chaque bande $[f_{min}, f_{max}]$ :

   
2. sélectionner les indices tels que $f_{min} \le f < f_{max}$ ;
3. extraire `band_psd` ;
4. calculer `sum`, `mean`, `max`, `min`, `median` ;
5. stocker les résultats dans un dictionnaire.

### Questions

1. Quelle bande doit dominer pour un signal sinusoïdal à 10 Hz ?
2. Pourquoi la PSD est-elle plus stable avec Welch qu'avec un simple spectre FFT ?
3. Que se passe-t-il si la bande sélectionnée ne contient aucune fréquence ?

### Réponses 

1. La bande alpha
2. Welch découpe le signal en sous-fenêtre et fait une moyenne de toutes les sous-fenêtres alors que FFT fait la moyenne de tout le signal d'un coup donc il est plus sensible aux valeurs aberrantes
3. Alors la moyenne est de 0

In [5]:
def compute_welch_psd(signal, fs=FS):
    nperseg = min(len(signal), fs * 2)
    freqs, psd = welch(signal, fs=fs, nperseg=nperseg)
    return freqs, psd


def compute_psd_band_features(signal, fs=FS):
    freqs, psd = compute_welch_psd(signal, fs)
    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        mask = (freqs >= fmin) & (freqs < fmax)
        band_psd = psd[mask]
        if len(band_psd) == 0:
            features[f"{band_name}_abs"]    = 0.0
            features[f"{band_name}_mean"]   = 0.0
            features[f"{band_name}_max"]    = 0.0
            features[f"{band_name}_min"]    = 0.0
            features[f"{band_name}_median"] = 0.0
        else:
            features[f"{band_name}_abs"]    = np.sum(band_psd)
            features[f"{band_name}_mean"]   = np.mean(band_psd)
            features[f"{band_name}_max"]    = np.max(band_psd)
            features[f"{band_name}_min"]    = np.min(band_psd)
            features[f"{band_name}_median"] = np.median(band_psd)
    return features


# Test sur les signaux synthétiques : la bande dominante doit correspondre à la fréquence du signal
for name, sig in synthetic_signals.items():
    feats = compute_psd_band_features(sig)
    dominant = max(EEG_BANDS.keys(), key=lambda b: feats[f"{b}_abs"])
    print(f"{name:15s} → bande dominante : {dominant}")

delta_2Hz       → bande dominante : delta
alpha_10Hz      → bande dominante : alpha
beta_20Hz       → bande dominante : beta
white_noise     → bande dominante : gamma
mixed           → bande dominante : theta


## 5. Entropie spectrale

### Principe théorique

L'entropie spectrale mesure la dispersion de l'énergie dans le domaine fréquentiel.

On calcule d'abord la PSD dans une bande, puis on la normalise pour obtenir une distribution :

$$
p_i = \frac{PSD_i}{\sum_j PSD_j}
$$

L'entropie de Shannon est ensuite :

$$
H = - \sum_i p_i \log(p_i)
$$


### Fonction Python utile

```python
from scipy.stats import entropy
```

`entropy(p)` calcule directement l'entropie de Shannon si `p` est une distribution normalisée.

### Questions

1. Pourquoi faut-il normaliser la PSD avant de calculer l'entropie ?
2. Quel signal devrait avoir l'entropie spectrale la plus élevée : une sinusoïde pure ou un bruit blanc ?
3. Pourquoi l'entropie spectrale peut-elle être utile pour caractériser la complexité d'un EEG ?

### Réponses

1. L'entropie de Shannon s'applique à une distribution de probabilité : chaque valeur doit être positive et leur somme doit valoir 1. La PSD brute est en µV²/Hz, pas en probabilité. La normalisation `p_i = PSD_i / sum(PSD)` transforme la PSD en distribution représentant la fraction d'énergie à chaque fréquence.

2. Le bruit blanc. Une sinusoïde pure concentre toute son énergie sur une seule fréquence : un seul `p_i` ≈ 1, tous les autres ≈ 0 → entropie très basse. Le bruit blanc répartit son énergie uniformément sur toutes les fréquences → entropie maximale.

3. Un EEG au repos est dominé par une bande (alpha) → énergie concentrée → entropie basse. Sous charge cognitive, le cerveau traite plusieurs types d'informations en parallèle → énergie dispersée sur plusieurs bandes → entropie plus élevée. L'entropie spectrale capture donc le degré de complexité de l'activité cérébrale.

In [ ]:
def compute_spectral_entropy_bands(signal, fs=FS):
    freqs, psd = compute_welch_psd(signal, fs)
    features = {}
    for band_name, (fmin, fmax) in EEG_BANDS.items():
        mask = (freqs >= fmin) & (freqs < fmax)
        band_psd = psd[mask]
        if len(band_psd) == 0 or np.sum(band_psd) == 0:
            features[f"{band_name}_entropy"] = 0.0
        else:
            p = band_psd / np.sum(band_psd)
            features[f"{band_name}_entropy"] = entropy(p)
    return features


# Test : le bruit blanc doit avoir l'entropie la plus élevée dans chaque bande
for name, sig in synthetic_signals.items():
    feats = compute_spectral_entropy_bands(sig)
    mean_entropy = np.mean(list(feats.values()))
    print(f"{name:15s} → entropie moyenne : {mean_entropy:.3f}")

## 6. Paramètres de Hjorth : mobility et complexity

Les paramètres de Hjorth sont des descripteurs temporels utilisés pour caractériser la dynamique d'un signal.

### Mobility

La mobility mesure approximativement la fréquence moyenne du signal :

$$
Mobility(x) = \sqrt{\frac{Var(\Delta x)}{Var(x)}}
$$

où $\Delta x$ est la dérivée discrète du signal, généralement calculée avec `np.diff(x)`.

### Complexity

La complexity mesure la variation de la mobility entre le signal et sa dérivée :

$$
Complexity(x) = \frac{Mobility(\Delta x)}{Mobility(x)}
$$


### Questions

1. Que vaut approximativement la variance d'un signal constant ?
2. Pourquoi faut-il gérer le cas où `Var(x)=0` ?
3. Entre une sinusoïde lisse et un bruit blanc, lequel devrait avoir une complexity plus élevée ?

### Réponses

1. La variance d'un signal constant est 0 : toutes les valeurs sont identiques, il n'y a aucune déviation par rapport à la moyenne.

2. Si `Var(x) = 0`, la formule `sqrt(Var(dx) / Var(x))` provoque une division par zéro, ce qui produit un `NaN` ou une erreur. Il faut retourner 0 dans ce cas pour éviter de propager des valeurs invalides.

3. Le bruit blanc. La complexity mesure la variation de la mobility entre le signal et sa dérivée. Une sinusoïde lisse a une dérivée régulière (cosinus) dont la mobility est similaire à celle du signal → complexity ≈ 1. Le bruit blanc a une dérivée encore plus irrégulière que le signal → complexity nettement supérieure à 1.

In [ ]:
def compute_hjorth(signal):
    x = np.array(signal, dtype=float)
    dx = np.diff(x)
    ddx = np.diff(dx)

    var_x = np.var(x)
    var_dx = np.var(dx)
    var_ddx = np.var(ddx)

    if var_x == 0:
        return {"hjorth_mobility": 0.0, "hjorth_complexity": 0.0}

    mobility = np.sqrt(var_dx / var_x)

    if var_dx == 0:
        complexity = 0.0
    else:
        mobility_dx = np.sqrt(var_ddx / var_dx)
        complexity = mobility_dx / mobility

    return {"hjorth_mobility": mobility, "hjorth_complexity": complexity}


# Test : le bruit blanc doit avoir la complexity la plus élevée
for name, sig in synthetic_signals.items():
    feats = compute_hjorth(sig)
    print(f"{name:15s} → mobility={feats['hjorth_mobility']:.4f}, complexity={feats['hjorth_complexity']:.4f}")

## 7. Complexité de Lempel-Ziv

### Principe théorique

La complexité de Lempel-Ziv mesure le nombre de motifs nouveaux rencontrés dans une séquence.

Comme l'algorithme s'applique à une séquence symbolique, un signal EEG réel doit d'abord être transformé en séquence binaire.

Une méthode simple consiste à binariser le signal par rapport à sa médiane :

$$
b[n] =
\begin{cases}
1, & x[n] > median(x) \\
0, & x[n] \le median(x)
\end{cases}
$$


### Paramètres importants

- seuil de binarisation : médiane ou moyenne ;
- normalisation de la complexité pour comparer des segments de même ou de différente longueur ;
- gestion des segments constants.

### Questions

1. Pourquoi faut-il binariser le signal avant de calculer Lempel-Ziv ?
2. Pourquoi la médiane est-elle un seuil intéressant ?
3. Quel signal devrait avoir une complexité plus élevée : une sinusoïde pure ou un bruit blanc ?
4. Pourquoi normaliser la complexité par la longueur de la séquence ?

### Réponses

1. L'algorithme de Lempel-Ziv est défini sur des séquences symboliques (alphabet fini). Un signal EEG est continu : deux valeurs identiques sont quasi-impossibles, donc chaque point serait un nouveau motif. La binarisation réduit l'alphabet à {0, 1} pour que l'algorithme détecte des répétitions de motifs.

2. La médiane partage le signal en deux moitiés égales, garantissant environ autant de 0 que de 1 dans la séquence binaire. Une séquence constante à 0 ou 1 aurait une complexité minimale, ce qui fausserait l'estimation.

3. Le bruit blanc. Une sinusoïde produit une séquence binaire très régulière (alternance prévisible de 0 et 1) → peu de nouveaux motifs → complexité basse. Le bruit blanc produit une séquence quasi-aléatoire → beaucoup de nouveaux motifs → complexité élevée.

4. Des segments plus longs auront mécaniquement plus de motifs distincts. La normalisation par `n / log2(n)` permet de comparer des segments de longueurs différentes sur une échelle commune entre 0 et 1.

In [ ]:
def lempel_ziv_complexity(signal):
    x = np.array(signal, dtype=float)
    binary = ''.join('1' if v > np.median(x) else '0' for v in x)
    n = len(binary)

    if n == 0:
        return {"lempel_ziv": 0.0}

    # Algorithme LZ76 : compte le nombre de sous-chaînes nouvelles
    sub_strings = set()
    c = 0
    i = 0
    while i < n:
        j = i + 1
        while j <= n:
            if binary[i:j] not in sub_strings:
                sub_strings.add(binary[i:j])
                c += 1
                i = j
                break
            j += 1
        else:
            break

    # Normalisation par la longueur théorique maximale
    lzc = c / (n / np.log2(n + 1e-10))
    return {"lempel_ziv": lzc}


# Test : le bruit blanc doit avoir la complexité LZ la plus élevée
for name, sig in synthetic_signals.items():
    feats = lempel_ziv_complexity(sig)
    print(f"{name:15s} → LZ complexity={feats['lempel_ziv']:.4f}")

## 8. Dimension fractale de Higuchi

### Principe théorique

La dimension fractale de Higuchi cherche à mesurer la complexité géométrique d'un signal temporel.

Un signal très lisse ressemble davantage à une courbe régulière. Un signal très irrégulier ou bruité présente une trajectoire plus complexe.

L'algorithme de Higuchi construit plusieurs sous-séquences avec différents pas $k$, mesure leur longueur moyenne $L(k)$, puis estime une pente dans un espace logarithmique.

### Paramètre important

- `kmax` : pas maximal testé.

Pour un segment de 10 secondes à 256 Hz, une valeur pédagogique simple est :

```python
kmax = 10
```

Une valeur trop faible peut donner une estimation instable ; une valeur trop élevée augmente le coût de calcul.

### Questions

1. Que cherche à mesurer la dimension fractale de Higuchi ?
2. Pourquoi un signal bruité peut-il avoir une dimension fractale plus élevée qu'une sinusoïde ?
3. Quel est le rôle du paramètre `kmax` ?
4. Pourquoi faut-il éviter de calculer un logarithme de zéro ?

### Réponses

1. La dimension fractale de Higuchi mesure la complexité géométrique d'un signal temporel : à quel point la courbe est "remplie" dans l'espace temps-amplitude. Une valeur proche de 1 indique un signal lisse (courbe régulière), une valeur proche de 2 indique un signal très irrégulier.

2. Un signal bruité présente des variations rapides à toutes les échelles → la longueur de la courbe augmente fortement quand on zoome → dimension fractale élevée. Une sinusoïde est géométriquement lisse, sa longueur augmente peu quand on zoome → dimension fractale proche de 1.

3. `kmax` est le pas maximal utilisé pour construire les sous-séquences. Il définit l'échelle jusqu'à laquelle on analyse la complexité. Trop petit : estimation instable (peu de points pour la régression). Trop grand : coût de calcul élevé sans gain d'information.

4. La régression se fait dans l'espace log-log : `log(L(k))` vs `log(k)`. Si `L(k) = 0` (signal constant sur une sous-séquence), `log(0)` vaut `-inf`, ce qui fausse la régression linéaire ou provoque une erreur numérique.

In [ ]:
def higuchi_fd(signal, kmax=10):
    x = np.array(signal, dtype=float)
    n = len(x)

    L = []
    ks = []

    for k in range(1, kmax + 1):
        Lk = []
        for m in range(1, k + 1):
            x_m = x[m - 1:n:k]
            n_m = len(x_m) - 1
            if n_m < 1:
                continue
            Lmk = np.sum(np.abs(np.diff(x_m))) * (n - 1) / (n_m * k * k)
            Lk.append(Lmk)
        if Lk:
            L.append(np.mean(Lk))
            ks.append(k)

    L = np.array(L)
    ks = np.array(ks, dtype=float)

    valid = L > 0
    if np.sum(valid) < 2:
        return {"higuchi_fd": 0.0}

    # Régression linéaire dans l'espace log-log : pente = -FD
    coeffs = np.polyfit(np.log(ks[valid]), np.log(L[valid]), 1)
    return {"higuchi_fd": -coeffs[0]}


# Test : le bruit blanc doit avoir la FD la plus élevée
for name, sig in synthetic_signals.items():
    feats = higuchi_fd(sig)
    print(f"{name:15s} → Higuchi FD={feats['higuchi_fd']:.4f}")

## 9. Statistiques temporelles du signal brut

Les statistiques temporelles simples fournissent des informations directes sur l'amplitude et la variabilité du signal.

Features demandées :

1. moyenne ;
2. minimum ;
3. maximum ;
4. médiane ;
5. variance ;
6. écart-type.

### Fonctions Python utiles

| Feature | Fonction NumPy |
|---|---|
| moyenne | `np.mean` |
| minimum | `np.min` |
| maximum | `np.max` |
| médiane | `np.median` |
| variance | `np.var` |
| écart-type | `np.std` |


In [ ]:
def compute_raw_features(signal):
    x = np.array(signal, dtype=float)
    return {
        "raw_mean":   np.mean(x),
        "raw_min":    np.min(x),
        "raw_max":    np.max(x),
        "raw_median": np.median(x),
        "raw_var":    np.var(x),
        "raw_std":    np.std(x),
    }


# Test rapide
for name, sig in synthetic_signals.items():
    feats = compute_raw_features(sig)
    print(f"{name:15s} → mean={feats['raw_mean']:.3f}, std={feats['raw_std']:.3f}")

## 10. Fonction complète d'extraction des 40 features EEG

À ce stade, on peut regrouper toutes les familles de features dans une seule fonction.

### Entrée

Un segment EEG 1D correspondant à :

- un canal ;
- une fenêtre de 10 secondes ;
- 2560 échantillons si `fs=256 Hz`.

### Sortie

Un dictionnaire de **40 features**.

### Organisation recommandée

1. convertir le signal en tableau NumPy ;
2. remplacer les valeurs manquantes par 0 ou par une stratégie décidée en amont ;
3. calculer les features PSD par bande ;
4. calculer les entropies spectrales ;
5. calculer Hjorth ;
6. calculer Lempel-Ziv ;
7. calculer Higuchi ;
8. calculer les statistiques temporelles ;
9. fusionner les dictionnaires.

### Questions

1. Pourquoi la fonction doit-elle retourner un dictionnaire plutôt qu'une simple liste ?
2. Pourquoi est-il important de conserver des noms de colonnes explicites ?
3. Combien de features doit retourner la fonction pour un canal ?
4. Si on a 4 canaux et qu'on concatène toutes les features, combien de features obtient-on par segment ?

### Réponses

1. Un dictionnaire avec des clés nommées permet de construire directement un DataFrame avec des colonnes identifiables. Une simple liste de 40 valeurs serait illisible : impossible de savoir quelle valeur correspond à quelle feature sans documentation externe.

2. Les noms explicites permettent d'interpréter les résultats, de réaliser des analyses de feature importance (quel canal, quelle bande, quel descripteur contribue le plus), et de débuguer facilement si une feature a une valeur aberrante.

3. 40 features par canal : 25 (PSD) + 5 (entropies) + 2 (Hjorth) + 1 (LZ) + 1 (Higuchi) + 6 (stats) = 40.

4. 4 canaux × 40 features = **160 features** par segment de 10 secondes.

In [ ]:
def extract_eeg_features(signal, fs=FS):
    x = np.array(signal, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)

    features = {}
    features.update(compute_psd_band_features(x, fs))
    features.update(compute_spectral_entropy_bands(x, fs))
    features.update(compute_hjorth(x))
    features.update(lempel_ziv_complexity(x))
    features.update(higuchi_fd(x))
    features.update(compute_raw_features(x))

    return features


# Vérification : doit retourner exactement 40 features
feats = extract_eeg_features(synthetic_signals["alpha_10Hz"])
print(f"Nombre de features : {len(feats)}")
assert len(feats) == 40, f"Attendu 40, obtenu {len(feats)}"
print("OK : 40 features")
print(list(feats.keys()))

## 11. Comparaison des features sur signaux connus

On applique maintenant la fonction complète aux signaux synthétiques.

### Objectif

Vérifier que :

- `alpha_10Hz` a une puissance alpha élevée ;
- `beta_20Hz` a une puissance beta élevée ;
- `white_noise` a une entropie spectrale élevée ;
- les features non linéaires augmentent généralement avec l'irrégularité.

In [ ]:
results = {}
for name, sig in synthetic_signals.items():
    results[name] = extract_eeg_features(sig)

df_synth = pd.DataFrame(results).T

# Colonnes à afficher pour vérifier la cohérence
cols_check = [
    "alpha_abs", "beta_abs", "theta_abs", "delta_abs",
    "alpha_entropy", "gamma_entropy",
    "hjorth_mobility", "hjorth_complexity",
    "lempel_ziv", "higuchi_fd",
]
print(df_synth[cols_check].round(4).to_string())

print("\nInterprétation attendue :")
print("- alpha_10Hz  : alpha_abs dominant")
print("- beta_20Hz   : beta_abs dominant")
print("- white_noise : entropies élevées, higuchi_fd élevé, lempel_ziv élevé")
print("- mixed       : theta/alpha tous les deux présents")

## 12. Application aux signaux EEG du dataset CL-Drive

Après les tests pédagogiques, les mêmes fonctions doivent être appliquées aux signaux EEG prétraités.

### Hypothèse de structure des fichiers

On suppose que les fichiers EEG prétraités sont des fichiers CSV contenant :

- une colonne `Timestamp` ;
- une colonne par canal EEG, par exemple `AF7`, `AF8`, `TP9`, `TP10`.

Exemple de structure :

| Timestamp | AF7 | AF8 | TP9 | TP10 |
|---:|---:|---:|---:|---:|
| 0.000 | ... | ... | ... | ... |
| 0.004 | ... | ... | ... | ... |

### Algorithme d'extraction sur un fichier

1. lire le fichier CSV avec `pd.read_csv` ;
2. identifier les colonnes EEG ;
3. découper le signal en fenêtres de 10 secondes ;
4. pour chaque fenêtre :
   - extraire les 2560 échantillons ;
   - pour chaque canal, calculer les 40 features ;
   - stocker les métadonnées : sujet, fichier, fenêtre, temps début, temps fin, canal ;
5. construire un `DataFrame` ;
6. sauvegarder le résultat en CSV.

### Question

Pourquoi faut-il conserver les colonnes `sujet`, `scénario`, `fenêtre`, `canal`, `temps début` et `temps fin` avec les features ?

### Réponse

Sans ces métadonnées, chaque ligne du DataFrame de features est un vecteur de 40 nombres anonymes. Il serait impossible de :
- savoir à quel sujet la ligne appartient (nécessaire pour LOSO et la normalisation par baseline) ;
- associer la fenêtre à son label PAAS (les labels sont dans un fichier séparé, l'association se fait par sujet + scénario + intervalle temporel) ;
- débuguer une valeur aberrante (on ne saurait pas quel fichier, quel canal ou quelle fenêtre est en cause) ;
- reproduire l'expérience (traçabilité complète de chaque observation).

In [ ]:
def extract_features_from_dataframe(df, fs=FS, window_sec=WINDOW_SEC, subject="unknown", scenario="unknown"):
    eeg_cols = [c for c in df.columns if c != "Timestamp"]
    window_samples = fs * window_sec
    n_samples = len(df)
    n_windows = n_samples // window_samples

    records = []
    for w in range(n_windows):
        start = w * window_samples
        end = start + window_samples
        t_start = df["Timestamp"].iloc[start] if "Timestamp" in df.columns else start / fs
        t_end = df["Timestamp"].iloc[end - 1] if "Timestamp" in df.columns else (end - 1) / fs

        for canal in eeg_cols:
            segment = df[canal].iloc[start:end].values
            feats = extract_eeg_features(segment, fs)
            feats["subject"] = subject
            feats["scenario"] = scenario
            feats["window"] = w
            feats["canal"] = canal
            feats["t_start"] = t_start
            feats["t_end"] = t_end
            records.append(feats)

    return pd.DataFrame(records)


print("Fonction extract_features_from_dataframe définie.")
print(f"Colonnes de sortie : 40 features + 6 métadonnées = 46 colonnes par ligne.")

## 13. Traitement par lot des fichiers EEG prétraités

Dans le projet, les fichiers prétraités peuvent être organisés par sujet.

Exemple :

```text
Data/
|----EEG/
    ├── Player_01/
    │   ├── filtered_scenario_1.csv
    │   ├── filtered_scenario_2.csv
    │   └── ...
    ├── Player_02/
    │   └── ...
|----EDA
|----ECG
|----Gaze
|----Labels
```

### Algorithme par lot

1. parcourir les dossiers sujets ;
2. sélectionner uniquement les fichiers `filtered_*.csv` ;
3. lire chaque fichier ;
4. extraire les features fenêtre par fenêtre ;
6. sauvegarder un fichier CSV de features par sujet.

### Remarque importante

Les labels PAAS ne se trouvent pas dans le même fichier que les signaux. Une étape d’association entre les features et les labels est donc nécessaire dans un second temps : il faut relier les scores de charge cognitive aux intervalles temporels correspondants.

In [ ]:
def batch_extract_eeg_features(base_path, output_path, fs=FS, window_sec=WINDOW_SEC):
    base_path = Path(base_path)
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)

    for subject_dir in sorted(base_path.iterdir()):
        if not subject_dir.is_dir():
            continue
        subject = subject_dir.name
        all_records = []

        for csv_file in sorted(subject_dir.glob("filtered_*.csv")):
            scenario = csv_file.stem.replace("filtered_", "")
            df = pd.read_csv(csv_file)
            df_feats = extract_features_from_dataframe(
                df, fs=fs, window_sec=window_sec,
                subject=subject, scenario=scenario,
            )
            all_records.append(df_feats)
            print(f"  {csv_file.name} → {len(df_feats)} lignes")

        if all_records:
            df_subject = pd.concat(all_records, ignore_index=True)
            out_file = output_path / f"{subject}_features.csv"
            df_subject.to_csv(out_file, index=False)
            print(f"{subject} → sauvegardé ({len(df_subject)} lignes) dans {out_file}\n")


print("Fonction batch_extract_eeg_features définie.")
print("Exemple d'utilisation :")
print('# batch_extract_eeg_features("Data/EEG", "Data/EEG_Features_10s")')

## 14. Vérifications qualité des features

Avant de passer à la classification, il faut vérifier la qualité de la matrice de features.

### Vérifications recommandées

1. nombre de lignes cohérent avec le nombre de fenêtres et de canaux ;
2. absence de valeurs manquantes ;
3. absence de valeurs infinies ;
4. ordre de grandeur plausible ;
5. nombre de features égal à 40 par canal ;
6. conservation des métadonnées utiles ;
7. possibilité d'associer ensuite chaque fenêtre à un label.

### Questions

1. Pourquoi des valeurs `NaN` peuvent-elles apparaître dans les features ?
2. Pourquoi des valeurs infinies peuvent-elles apparaître ?
3. Que doit-on faire si un segment contient trop de valeurs manquantes ?
4. Pourquoi faut-il éviter de normaliser les features avant la séparation train/test ?

### Réponses

1. Un segment peut contenir des `NaN` si le capteur a perdu la connexion Bluetooth (cas mentionné dans le papier). Une division par une variance nulle (signal constant) peut aussi produire un `NaN` si le cas n'est pas géré.

2. Des valeurs infinies peuvent apparaître lors d'une division par zéro non protégée (ex. normalisation LZ avec `log2(0)`), ou si un segment contient des `inf` hérités du prétraitement.

3. Il faut exclure le segment entier plutôt que de l'imputer : un segment avec trop de valeurs manquantes ne représente pas une activité cérébrale réelle. Le papier exclut d'ailleurs les segments avec données manquantes pour cette raison.

4. Si on normalise (z-score, min-max) sur l'ensemble du dataset avant de séparer, les statistiques du train set "fuient" dans le test set. Le modèle bénéficie d'informations qu'il n'aurait pas en production. Il faut fitter le scaler uniquement sur le train set, puis l'appliquer au test set.

In [ ]:
def check_features_quality(df, expected_features=40):
    meta_cols = ["subject", "scenario", "window", "canal", "t_start", "t_end"]
    feature_cols = [c for c in df.columns if c not in meta_cols]

    print("=== Contrôle qualité des features ===\n")
    print(f"Lignes       : {len(df)}")
    print(f"Features     : {len(feature_cols)}", end="")
    if len(feature_cols) != expected_features:
        print(f"  ⚠ attendu {expected_features}")
    else:
        print("  OK")

    n_nan = df[feature_cols].isna().sum().sum()
    print(f"NaN          : {n_nan}", end="")
    if n_nan > 0:
        cols_nan = df[feature_cols].isna().sum()
        print(f"  ⚠ colonnes : {cols_nan[cols_nan > 0].to_dict()}")
    else:
        print("  OK")

    n_inf = np.isinf(df[feature_cols].select_dtypes(include=np.number).values).sum()
    print(f"Inf          : {n_inf}", end="")
    print("  OK" if n_inf == 0 else "  ⚠")

    print("\nOrdre de grandeur (5 premières features) :")
    print(df[feature_cols[:5]].describe().round(4).to_string())

    return n_nan == 0 and n_inf == 0


# Test sur les signaux synthétiques
df_test = pd.DataFrame([extract_eeg_features(sig) for sig in synthetic_signals.values()])
check_features_quality(df_test)